# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, leveraging the [mlcroissant](https://mlcommons.github.io/croissant/) library for standardized access and exploration.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (schema and metadata)
dataset = mlc.Dataset(croissant_url)

# Display key metadata fields
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets, and for each, print the available fields and their `@id`.

The Croissant schema uniquely identifies each record set, field, and column by `@id`. Be sure to use these for further data access.

In [ ]:
# List all record sets
print("Available record sets:")
record_sets = dataset.metadata.recordSet
for rs in record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs['@id']}")
    print("  Fields (columns):")
    # Fields/columns may be specified in both 'field' (as Field objects) and/or 'column' under the RecordSet
    if hasattr(rs, 'field') and rs.field:
        for field in rs.field:
            print(f"    - {field.name} (@id={field['@id']}) [dataType={field.dataType if hasattr(field, 'dataType') else 'unknown'}]")
    if hasattr(rs, 'column') and rs.column:
        for col in rs.column:
            # A Column might not have `name`; fall back to @id
            print(f"    - Column: {getattr(col, 'name', '[no name]')} (@id={col['@id']})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

Here, we automatically detect and use the first (main) record set.

In [ ]:
# Choose record set to analyze (the main or first record set)
main_record_set = record_sets[0]  # Use the first available record set
main_record_set_id = main_record_set['@id']

print(f"Main RecordSet chosen: {main_record_set.name} (ID: {main_record_set_id})")

# Load records from the chosen record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

# Display the fields/columns available
print(f"Available columns in DataFrame:")
print(df.columns.tolist())

# Show a preview
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll demonstrate by selecting a numeric field and a grouping field by their `@id`.

In [ ]:
# Suggest likely numeric and group fields based on DataFrame columns
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or df[col].str.replace('.', '', 1).str.isnumeric().all()]
group_candidates = [col for col in df.columns if df[col].nunique() < 10 and df[col].dtype == object]
print(f"Numeric field candidates: {numeric_candidates}")
print(f"Grouping field candidates: {group_candidates}")

# For this dataset, typical relevant numeric fields might include 'Coverage', 'MW', 'Spectral_Count', etc.
# You can adjust these fields below based on the real dataset columns.

# Choose a numeric and grouping field by @id/column name
# Example: numeric_field = '@id:coverage'
# Example: group_field = '@id:sample_group' or similar
numeric_field = numeric_candidates[0] if len(numeric_candidates) else None
group_field = group_candidates[0] if len(group_candidates) else None
print(f"Using numeric_field: {numeric_field}")
print(f"Using group_field: {group_field}")

if numeric_field:
    # Convert to numeric if needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()  # Use mean as threshold for demo
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print('No numeric field found for EDA.')

## 5. Visualization
Visualize distributions or relationships in the data. For example, plot the distribution of the main numeric field and, if available, show group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
We demonstrated how to load, explore, and process the FAIR² mass spectrometry dataset using `mlcroissant`. By referencing fields and record sets with their `@id`, you ensure reproducible and schema-aligned data processing workflows suitable for research and downstream ML tasks.

**Always refer to the Croissant schema for authoritative mapping between column names and their `@id` values. For advanced analyses, extend the EDA and visualization sections to suit your study's needs.**